## 1. A 做市与 B 周期持仓（v4）

A+B 合计净仓位上限 ±200 股，包含同向挂单。B 启动时先用近期历史拟合初始 180 秒周期，交易期间持续验证并调整周期、窗口、偏移、振幅、相位和信号权重，优先一次建立可用目标仓位；A 每笔新增 100 股，减仓可报全部剩余量。


In [ ]:
import logging
from dataclasses import asdict
import math
from pathlib import Path
import sys
import time


def find_workspace():
    anchors = [Path.cwd(), Path('/home/workspace')]
    if '__file__' in globals():
        anchors.insert(0, Path(__file__).resolve().parent)
    for anchor in anchors:
        for parent in (anchor, *anchor.parents):
            for candidate in (parent, parent / 'your_optiver_workspace'):
                if ((candidate / 'common' / 'trade_logger.py').is_file()
                        and (candidate / 'stock_market_making' / 'recording' / 'strategy_recording.py').is_file()):
                    return candidate.resolve()
    raise FileNotFoundError(
        'Cannot find the strategy helpers. Keep stock_market_making/ and common/ '
        'under your_optiver_workspace/ when copying this strategy.')


WORKSPACE_ROOT = find_workspace()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))
from stock_market_making.recording.strategy_recording import RecordedExchange, StrategyRecorder
__package__ = 'stock_market_making.strategies.baseline-refine'
from .order_execution import LimitedExchange, QuoteManager
from stock_market_making.quote_helpers import external_price_book
from .cycle_signal import (
    CycleSettings, CycleSignal, usable_book, apply_cycle_quote,
)
from stock_market_making.recording.record_philips_prices import PhilipsPriceRecorder
from stock_market_making.recording.storage import MARKET_DIR, RUNS_DIR, RunStorage
from .quote_protection import ProtectionSettings, protect_quote
from .cycle_position import CyclePosition
from .cycle_history import bootstrap_cycle


## 2. 参数

B 目标最多 100 股；建仓价改善外部同侧最优价 1 tick。止盈取加权预测空间的 80%，再留 2 ticks；止损为逆向 60 ticks 持续 3 秒。退出先改善最优价，2 秒未平完则跨到对手盘 10 ticks 内，全部剩余量持续退出。


In [ ]:
TRADE_INSTRUMENTS = {'PHILIPS_A', 'PHILIPS_B'}
EXCHANGE_POSITION_LIMIT = 100  # Optibook hard per-instrument limit.
POSITION_LIMIT = EXCHANGE_POSITION_LIMIT
NET_POSITION_LIMIT = 200  # A + B, including worst-case same-side resting orders.
SOFT_LIMIT = 100
ORDER_VOLUME = 100  # A new inventory; B requests its entire remaining target.
VWAP_HALF_LIFE_TICKS = 5

CYCLE_SETTINGS = CycleSettings(
    enabled=True, period_seconds=180.0, history_seconds=720.0,
    min_fit_seconds=30.0, ramp_seconds=180.0, horizon_seconds=45.0,
    adaptive=True, min_period_seconds=120.0, max_period_seconds=240.0,
    period_step_seconds=5.0, fast_history_seconds=360.0,
    switch_improvement=0.2, switch_confirmations=3, switch_cooldown_seconds=30.0,
)
INVENTORY_SCALE = 100
INVENTORY_SKEW_TICKS = 2.0
SPREAD_MULTIPLIER = 1.0
MIN_HALF_SPREAD_TICKS = 1.0
MARKOUT_HORIZONS = (1, 3, 5, 15, 30, 60)
MARKOUT_TIMEOUT_SECONDS = 90
STRATEGY_VERSION = 'baseline_refine_v4'
CYCLE_PRICE_SHIFT_ENABLED = False
B_PROTECTION = ProtectionSettings(max_increasing_volume=100)
B_POSITION_SETTINGS = dict(
    target_lots=100, reference_hold_seconds=45, entry_window_seconds=10,
    retry_seconds=1, cooldown_seconds=2,
    take_profit_fraction=0.8, take_profit_buffer_ticks=2,
    stop_ticks=60, stop_confirmation_seconds=3,
    exit_cross_after_seconds=2, exit_sweep_ticks=10,
)
MAX_OUTSTANDING_VOLUME = 200
MAX_UPDATES_PER_SECOND = 200
LOG_DIR = RUNS_DIR / 'baseline-refine'
PRICE_DATA_DIR = MARKET_DIR


## 3. 报价与库存

B 冻结方向和止盈价；45 秒仅为持仓参考，超过后不触发退出。未成交计划结束后 1 秒重试，完整退出后 2 秒冷却；达到止盈价可提前退出；部分成交不重置退出计时。宽价差只能阻止新增仓位，不能阻止已有仓位退出。净额度和挂单容量在实际发送前重新核对。


In [ ]:
def calculate_quote(book, position, tick_size):
    """Inventory-aware prices and target remaining sizes, using external depth."""
    if not book or not book.bids or not book.asks:
        return None
    if not math.isfinite(tick_size) or tick_size <= 0:
        return None
    parameters = (INVENTORY_SCALE, INVENTORY_SKEW_TICKS, SPREAD_MULTIPLIER,
                  MIN_HALF_SPREAD_TICKS, VWAP_HALF_LIFE_TICKS)
    if (not all(math.isfinite(value) for value in parameters)
            or INVENTORY_SCALE <= 0 or SOFT_LIMIT <= 0 or SOFT_LIMIT > POSITION_LIMIT
            or INVENTORY_SKEW_TICKS < 0 or SPREAD_MULTIPLIER <= 0
            or MIN_HALF_SPREAD_TICKS <= 0 or VWAP_HALF_LIFE_TICKS <= 0):
        raise ValueError('Invalid inventory or spread settings')
    bids, asks = book.bids, book.asks
    if (any(not math.isfinite(level.price) or level.price <= 0
            or not math.isfinite(level.volume) or level.volume <= 0
            for level in (*bids, *asks))
            or bids[0].price >= asks[0].price):
        return None
    bid_weights = [level.volume * 0.5 ** (
        (bids[0].price - level.price) / tick_size / VWAP_HALF_LIFE_TICKS)
        for level in bids]
    ask_weights = [level.volume * 0.5 ** (
        (level.price - asks[0].price) / tick_size / VWAP_HALF_LIFE_TICKS)
        for level in asks]
    bid_vwap = sum(level.price * weight for level, weight in zip(bids, bid_weights)) / sum(bid_weights)
    ask_vwap = sum(level.price * weight for level, weight in zip(asks, ask_weights)) / sum(ask_weights)
    fair_value = (bid_vwap + ask_vwap) / 2

    inventory_adjustment = INVENTORY_SKEW_TICKS * tick_size * position / INVENTORY_SCALE
    center = fair_value - inventory_adjustment
    half_spread = max(MIN_HALF_SPREAD_TICKS * tick_size,
                      SPREAD_MULTIPLIER * (asks[0].price - bids[0].price) / 2)

    # Compete at the touch. On a two-tick spread both improvements would
    # meet: give the inventory-reducing side priority instead of self-crossing.
    bid_price = round(min(bids[0].price + tick_size, asks[0].price - tick_size), 10)
    ask_price = round(max(asks[0].price - tick_size, bids[0].price + tick_size), 10)
    if bid_price >= ask_price:
        if position > 0 or (position == 0 and center < (bids[0].price + asks[0].price) / 2):
            bid_price = bids[0].price
        else:
            ask_price = asks[0].price
    buy_volume = min(ORDER_VOLUME, max(0, POSITION_LIMIT - position))
    sell_volume = min(ORDER_VOLUME, max(0, POSITION_LIMIT + position))
    if position < 0:
        buy_volume = abs(position)
    elif position > 0:
        sell_volume = position
    if bid_price <= 0:
        buy_volume = 0
    if ask_price <= 0:
        sell_volume = 0
    return dict(fair_value=fair_value, center=center,
                bid_price=bid_price, ask_price=ask_price,
                buy_volume=buy_volume, sell_volume=sell_volume,
                inventory_adjustment=inventory_adjustment, half_spread=half_spread,
                book_scope='excluding_own_orders', reduce_bid=position < 0, reduce_ask=position > 0)


def plan_quote(book, position, tick, instrument_id, signal, b_position, now):
    """Single quote pipeline shared by the live loop and offline validation."""
    quote = calculate_quote(book, position, tick)
    quote = apply_cycle_quote(quote, book, position, tick, instrument_id,
                              signal, CYCLE_SETTINGS, SOFT_LIMIT,
                              apply_price_shift=CYCLE_PRICE_SHIFT_ENABLED)
    quote = protect_quote(quote, book, position, tick, instrument_id, B_PROTECTION)
    if quote is None:
        return None
    if instrument_id == 'PHILIPS_B':
        return b_position.apply(quote, book, position, tick, now)
    if book.asks[0].price - book.bids[0].price > CYCLE_SETTINGS.max_spread_ticks * tick:
        quote['reduce_only'] = True
        if position >= 0:
            quote['buy_volume'] = 0
        if position <= 0:
            quote['sell_volume'] = 0
    return quote


## 4. 单连接主循环

做市、周期学习、CSV 和成交日志共用一个 Exchange 连接。周期模块接收扣除自身挂单后的盘口，不消费成交流。

`QuoteManager` 继续维护挂单、核对实际仓位并执行原有额度限制。报价日志的 `cycle_shift` 和 `cycle` 记录停用原因、`fit_weight`、`fit_rmse`、`fit_r2` 和预测价差变化；每次重拟合另有 `cycle_model_update` 日志，记录周期、窗口、候选确认、切换和到期预测误差；`fit_weight` 综合样本内拟合、时间顺序验证与到期预测表现，样本内拟合质量不是预测准确率。停止撤单但仓位保留，不自动清仓。


In [ ]:
def main():
    from optibook.synchronous_client import Exchange

    logging.getLogger('client').setLevel('ERROR')
    exchange = Exchange(max_nr_trade_history=10000)
    recorder = price_recorder = None
    try:
        exchange.connect()
        instruments = exchange.get_tradable_instruments()
        monitor_ids = tuple(instruments)
        missing_ids = TRADE_INSTRUMENTS.difference(instruments)
        if missing_ids or not TRADE_INSTRUMENTS:
            raise ValueError(f'Trading selection is empty or unavailable: {sorted(missing_ids)}')
        trade_ids = tuple(iid for iid in ('PHILIPS_B', 'PHILIPS_A') if iid in TRADE_INSTRUMENTS)

        run_storage = RunStorage('baseline-refine', directory=LOG_DIR)
        recorder = StrategyRecorder(
            exchange, monitor_ids, run_storage.directory, horizons=MARKOUT_HORIZONS,
            markout_timeout_seconds=MARKOUT_TIMEOUT_SECONDS)
        print(f'Strategy: {STRATEGY_VERSION}; B protection: {asdict(B_PROTECTION)};               f'per-instrument limit: {EXCHANGE_POSITION_LIMIT}; net limit: {NET_POSITION_LIMIT}', flush=True)
        price_recorder = PhilipsPriceRecorder(exchange, PRICE_DATA_DIR)
        run_storage.link_events(recorder.path)
        run_storage.link_market(price_recorder.directory)
        print(f'Run manifest: {run_storage.path}', flush=True)
        print(f'PHILIPS A/B CSV folder: {price_recorder.directory}', flush=True)
        exchange = LimitedExchange(
            exchange, max_outstanding_volume=MAX_OUTSTANDING_VOLUME,
            max_updates_per_second=MAX_UPDATES_PER_SECOND,
            position_limit=EXCHANGE_POSITION_LIMIT)
        exchange = RecordedExchange(exchange, recorder)
        quote_manager = QuoteManager(exchange, position_limit=POSITION_LIMIT, soft_limit=SOFT_LIMIT,
                                     net_position_limit=NET_POSITION_LIMIT, net_symbols=('PHILIPS_A', 'PHILIPS_B'))
        print(f'Log file: {recorder.path}', flush=True)
        print(f'Trading: {trade_ids}; monitoring and cancelling: {monitor_ids}', flush=True)
        run_config = dict(strategy_version=STRATEGY_VERSION,
                       cycle_price_shift_enabled=CYCLE_PRICE_SHIFT_ENABLED,
                       b_protection=asdict(B_PROTECTION),
                       b_position_settings=B_POSITION_SETTINGS,
                       markout_timeout_seconds=MARKOUT_TIMEOUT_SECONDS,
                       position_limit=POSITION_LIMIT, exchange_position_limit=EXCHANGE_POSITION_LIMIT,
                       net_position_limit=NET_POSITION_LIMIT,
                       net_position_scope="PHILIPS_A + PHILIPS_B", soft_limit=SOFT_LIMIT,
                       order_volume=ORDER_VOLUME, half_life_ticks=VWAP_HALF_LIFE_TICKS,
                       inventory_scale=INVENTORY_SCALE, inventory_skew_ticks=INVENTORY_SKEW_TICKS,
                       spread_multiplier=SPREAD_MULTIPLIER, min_half_spread_ticks=MIN_HALF_SPREAD_TICKS,
                       markout_horizons=MARKOUT_HORIZONS,
                       max_outstanding_volume=MAX_OUTSTANDING_VOLUME,
                       max_updates_per_second=MAX_UPDATES_PER_SECOND,
                       trade_ids=trade_ids, monitor_ids=monitor_ids,
                       cycle_settings=asdict(CYCLE_SETTINGS), cycle_reference='PHILIPS_A')
        run_storage.update(config=run_config)
        recorder.event('settings', **run_config)

        cycle_model = CycleSignal(CYCLE_SETTINGS)
        b_position = CyclePosition(**B_POSITION_SETTINGS)
        tick_sizes = {iid: info.tick_size for iid, info in instruments.items()}
        bootstrap_pending = True
        history_retry_at = -float('inf')
        last_model_fit = None
        last_error_print = -float('inf')
        while exchange.is_connected():
            cycle_start = time.monotonic()
            try:
                recorder.sample()
                price_recorder.sample()  # 同一连接，每轮记录 PHILIPS A/B 行情。

                # Non-traded instruments stay monitored and have no resting orders.
                for instrument_id in monitor_ids:
                    if instrument_id not in trade_ids:
                        quote_manager.reconcile(instrument_id, None)

                if bootstrap_pending or (cycle_model.needs_bootstrap and time.monotonic() >= history_retry_at):
                    bootstrap_report = bootstrap_cycle(
                        cycle_model, price_recorder.exchange, PRICE_DATA_DIR,
                        tick_sizes, time.monotonic(), time.time(),
                        scan_recordings=bootstrap_pending)
                    recorder.event('cycle_bootstrap', **bootstrap_report)
                    print('Cycle history:', bootstrap_report, flush=True)
                    bootstrap_pending = False
                    history_retry_at = time.monotonic() + CYCLE_SETTINGS.refit_seconds

                # Model and recorders share one connection; no additional trade polling.
                external_books = {}
                signal_ids = tuple(dict.fromkeys((*trade_ids, 'PHILIPS_A', 'PHILIPS_B')))
                for instrument_id in signal_ids:
                    if instrument_id not in instruments:
                        continue
                    raw_book = exchange.get_last_price_book(instrument_id)
                    own_orders = exchange.get_outstanding_orders(instrument_id)
                    tick_size = tick_sizes[instrument_id]
                    external_books[instrument_id] = (
                        external_price_book(raw_book, own_orders, tick_size)
                        if usable_book(raw_book, tick_size, time.time(), CYCLE_SETTINGS, check_spread=False) else None
                    )
                signal = cycle_model.observe(external_books, tick_sizes, time.monotonic(), time.time())
                if cycle_model.last_fit != last_model_fit:
                    recorder.event('cycle_model_update', **signal)
                    last_model_fit = cycle_model.last_fit
                    if signal['adaptation']['state'] == 'changed':
                        print('Cycle model changed:', signal['adaptation'],
                              'period=', signal['period_seconds'],
                              'window=', signal['window_seconds'], flush=True)
                for instrument_id in trade_ids:
                    tick_size = tick_sizes[instrument_id]
                    book = external_books.get(instrument_id)
                    # Repricing the previous instrument may take time.
                    if not usable_book(book, tick_size, time.time(), CYCLE_SETTINGS, check_spread=False):
                        quote_manager.reconcile(instrument_id, None)
                        recorder.event('skip_quote', instrument=instrument_id,
                                       reason='stale or invalid external book')
                        continue
                    position = exchange.get_positions()[instrument_id]
                    signal = cycle_model.observe(
                        external_books, tick_sizes, time.monotonic(), time.time()
                    )
                    quote = plan_quote(book, position, tick_size, instrument_id,
                                       signal, b_position, time.monotonic())
                    if quote is None:
                        quote_manager.reconcile(instrument_id, None)
                        continue
                    account_positions = exchange.get_positions()
                    quote['net_position'] = sum(account_positions[iid] for iid in ('PHILIPS_A', 'PHILIPS_B'))
                    quote['net_position_limit'] = NET_POSITION_LIMIT
                    recorder.quote(instrument_id, book, position, quote)
                    result = quote_manager.reconcile(instrument_id, quote)
                    recorder.event('quote_reconciled', instrument=instrument_id, result=result)

            except Exception as error:
                cycle_model.reset()
                last_model_fit = None
                bootstrap_pending = True
                recorder.event('cycle_error', error=str(error))
                if time.monotonic() - last_error_print >= 10:
                    print('Cycle stopped (further errors go to the log):', error, flush=True)
                    last_error_print = time.monotonic()
                for instrument_id in monitor_ids:
                    try:
                        exchange.delete_orders(instrument_id, reason='cycle error')
                    except Exception as cancel_error:
                        recorder.event('cancel_error', instrument=instrument_id, error=str(cancel_error))
                    finally:
                        time.sleep(0.1)
            finally:
                if exchange.is_connected():
                    recorder.sample()
                recorder.event('cycle_work_finished', seconds=time.monotonic() - cycle_start)
                time.sleep(0.5)
    except KeyboardInterrupt:
        pass
    finally:
        try:
            if price_recorder is not None:
                try:
                    if exchange.is_connected():
                        price_recorder.drain_trades()
                finally:
                    price_recorder.close()
        finally:
            try:
                # Disconnect removes orders; existing positions remain.
                exchange.disconnect()
            finally:
                if recorder is not None:
                    recorder.close()


## 5. 运行

在此目录使用普通脚本：`python run.py --live`。脚本只加载标记为 `strategy` 的定义和参数单元，然后调用一次 `main()`，不会运行分析单元。Ctrl+C 停止，修改 notebook 参数后重新启动脚本。

下方保留原有手动启动单元。离线查看和分析时不要执行它，也不要 Run All。每个团队只能有一个连接，不要同时运行其他策略或独立采集程序。


In [ ]:
# 运行此单元格即开始交易；点击 Interrupt / 中断内核停止。
print('Starting trading. Use Interrupt to stop.', flush=True)
main()
print('Trading stopped. Existing positions remain.', flush=True)


## 6. 查看已有成交日志

此单元只读取日志。先运行导入和参数单元即可使用。markout 是成交后价格变化诊断，不是完整已实现收益。


In [ ]:
from common.summarize_trade_log import summarize

log_files = list(LOG_DIR.rglob("trading_*.jsonl"))
legacy_log_dir = WORKSPACE_ROOT / "stock_market_making" / "logs"
log_files.extend(legacy_log_dir.glob("trading_*.jsonl"))
if log_files:
    latest_log = max(log_files, key=lambda path: path.stat().st_mtime_ns)
    print("Log:", latest_log)
    summarize(latest_log)
else:
    print("No log files yet.")
